## (old)Propagating synthetic big wave

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, FFMpegWriter
from pathlib import Path


# ==================================================
# Settings
# ==================================================
OUT_DIR = Path("propagating_wave_outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

NX = 250
NY = 250

DX_KM = 1.0
DY_KM = 1.0

DT_SECONDS = 30
TOTAL_TIME_SECONDS = 3600*4

SPEED_MS = 150.0
SPEED_KMS = SPEED_MS / 1000.0

WAVE_LAMBDA_KM = 800.0
WAVE_ANGLE_DEG = 15.0

WAVE_AMP = 1.0

# Large envelope so the wave fills most/all of the FOV
WAVE_SIGMA_X_KM = 1200.0
WAVE_SIGMA_Y_KM = 1200.0

NOISE_STD = 0.02
RANDOM_SEED = 1

FPS = 20
DPI = 200

CMAP = "RdBu_r"


# ==================================================
# Coordinate grid
# ==================================================
x_km = np.arange(NX) * DX_KM
y_km = np.arange(NY) * DY_KM
x2d, y2d = np.meshgrid(x_km, y_km)

x0 = 0.5 * (NX - 1) * DX_KM
y0 = 0.5 * (NY - 1) * DY_KM

times_s = np.arange(0, TOTAL_TIME_SECONDS + DT_SECONDS, DT_SECONDS)
times_min = times_s / 60.0
nt = len(times_s)

theta = np.deg2rad(WAVE_ANGLE_DEG)

# Propagation direction coordinate
s_km = (x2d - x0) * np.cos(theta) + (y2d - y0) * np.sin(theta)

# Envelope
envelope = np.exp(
    -0.5 * (
        ((x2d - x0) / WAVE_SIGMA_X_KM) ** 2
        + ((y2d - y0) / WAVE_SIGMA_Y_KM) ** 2
    )
)

k = 2.0 * np.pi / WAVE_LAMBDA_KM


# ==================================================
# Create time series
# ==================================================
rng = np.random.default_rng(RANDOM_SEED)

wave_cube = np.zeros((nt, NY, NX), dtype=np.float32)

for it, t in enumerate(times_s):
    displacement_km = SPEED_KMS * t

    phase = k * (s_km - displacement_km)

    field = WAVE_AMP * envelope * np.cos(phase)
    field += rng.normal(0.0, NOISE_STD, size=field.shape)

    wave_cube[it] = field.astype(np.float32)


# Consistent color limits
vmax = np.nanpercentile(np.abs(wave_cube), 99)
vmin = -vmax


# ==================================================
# Animation
# ==================================================
fig, ax = plt.subplots(figsize=(7, 6))

im = ax.imshow(
    wave_cube[0],
    origin="upper",
    cmap=CMAP,
    vmin=vmin,
    vmax=vmax,
    extent=[x_km.min(), x_km.max(), y_km.max(), y_km.min()],
)

cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label("Synthetic wave amplitude")

title = ax.set_title("")
ax.set_xlabel("E-W distance (km)")
ax.set_ylabel("N-S distance (km)")
ax.set_aspect("equal")


def update(frame):
    im.set_data(wave_cube[frame])
    title.set_text(
        f"Propagating Synthetic Wave | "
        f"t = {times_min[frame]:.1f} min | "
        f"speed = {SPEED_MS:.0f} m/s"
    )
    return im, title


ani = FuncAnimation(
    fig,
    update,
    frames=nt,
    interval=1000 / FPS,
    blit=False,
)

out_mp4 = OUT_DIR / "propagating_wave_animation.mp4"

writer = FFMpegWriter(fps=FPS, bitrate=3000)
ani.save(out_mp4, writer=writer, dpi=DPI)

plt.close(fig)


# ==================================================
# Keograms
# ==================================================
center_x_idx = NX // 2
center_y_idx = NY // 2

# N-S keogram: vertical slice through center x
keogram_ns = wave_cube[:, :, center_x_idx]

# E-W keogram: horizontal slice through center y
keogram_ew = wave_cube[:, center_y_idx, :]


# --------------------------
# N-S keogram
# --------------------------
fig_ns, ax_ns = plt.subplots(figsize=(9, 5))

im_ns = ax_ns.imshow(
    keogram_ns.T,
    origin="lower",
    aspect="auto",
    cmap=CMAP,
    vmin=vmin,
    vmax=vmax,
    extent=[
        times_min.min(),
        times_min.max(),
        y_km.min(),
        y_km.max(),
    ],
)

cbar_ns = fig_ns.colorbar(im_ns, ax=ax_ns, fraction=0.046, pad=0.04)
cbar_ns.set_label("Synthetic wave amplitude")

ax_ns.set_title("N-S Keogram")
ax_ns.set_xlabel("Time (min)")
ax_ns.set_ylabel("N-S distance (km)")

plt.tight_layout()

out_ns = OUT_DIR / "keogram_NS.png"
fig_ns.savefig(out_ns, dpi=DPI, bbox_inches="tight")
plt.close(fig_ns)


# --------------------------
# E-W keogram
# --------------------------
fig_ew, ax_ew = plt.subplots(figsize=(9, 5))

im_ew = ax_ew.imshow(
    keogram_ew.T,
    origin="lower",
    aspect="auto",
    cmap=CMAP,
    vmin=vmin,
    vmax=vmax,
    extent=[
        times_min.min(),
        times_min.max(),
        x_km.min(),
        x_km.max(),
    ],
)

cbar_ew = fig_ew.colorbar(im_ew, ax=ax_ew, fraction=0.046, pad=0.04)
cbar_ew.set_label("Synthetic wave amplitude")

ax_ew.set_title("E-W Keogram")
ax_ew.set_xlabel("Time (min)")
ax_ew.set_ylabel("E-W distance (km)")

plt.tight_layout()

out_ew = OUT_DIR / "keogram_EW.png"
fig_ew.savefig(out_ew, dpi=DPI, bbox_inches="tight")
plt.close(fig_ew)

np.save(OUT_DIR / "keogram_NS.npy", keogram_ns)
np.save(OUT_DIR / "keogram_EW.npy", keogram_ew)
np.save(OUT_DIR / "times_s.npy", times_s)
np.save(OUT_DIR / "x_km.npy", x_km)
np.save(OUT_DIR / "y_km.npy", y_km)

# ==================================================
# Save one final frame too
# ==================================================
fig_last, ax_last = plt.subplots(figsize=(7, 6))

im_last = ax_last.imshow(
    wave_cube[-1],
    origin="upper",
    cmap=CMAP,
    vmin=vmin,
    vmax=vmax,
    extent=[x_km.min(), x_km.max(), y_km.max(), y_km.min()],
)

cbar_last = fig_last.colorbar(im_last, ax=ax_last, fraction=0.046, pad=0.04)
cbar_last.set_label("Synthetic wave amplitude")

ax_last.set_title("Final Frame: Propagating Synthetic Wave")
ax_last.set_xlabel("E-W distance (km)")
ax_last.set_ylabel("N-S distance (km)")
ax_last.set_aspect("equal")

plt.tight_layout()

out_last = OUT_DIR / "final_frame.png"
fig_last.savefig(out_last, dpi=DPI, bbox_inches="tight")
plt.close(fig_last)


print("Saved:")
print(out_mp4)
print(out_ns)
print(out_ew)
print(out_last)

Saved:
propagating_wave_outputs\propagating_wave_animation.mp4
propagating_wave_outputs\keogram_NS.png
propagating_wave_outputs\keogram_EW.png
propagating_wave_outputs\final_frame.png


## (old) 7 keograms

In [8]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, FFMpegWriter
from pathlib import Path
from scipy.ndimage import map_coordinates


# ==================================================
# Settings
# ==================================================
OUT_DIR = Path("propagating_wave_outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

NX = 250
NY = 250

DX_KM = 1.0
DY_KM = 1.0

DT_SECONDS = 30
TOTAL_TIME_SECONDS = 3600 * 4

SPEED_MS = 150.0
SPEED_KMS = SPEED_MS / 1000.0

WAVE_LAMBDA_KM = 400.0
WAVE_ANGLE_DEG = 15.0

WAVE_AMP = 1.0

# Keep large envelope
WAVE_SIGMA_X_KM = 800.0
WAVE_SIGMA_Y_KM = 800.0

NOISE_STD = 0.02
RANDOM_SEED = 1

FPS = 20
DPI = 200

CMAP = "RdBu_r"

# Directional keograms
KEOGRAM_ANGLES_DEG = np.arange(0, 166, 15)
KEOGRAM_WIDTH_PIXELS = 3


# ==================================================
# Coordinate grid
# ==================================================
x_km = np.arange(NX) * DX_KM
y_km = np.arange(NY) * DY_KM
x2d, y2d = np.meshgrid(x_km, y_km)

x0 = 0.5 * (NX - 1) * DX_KM
y0 = 0.5 * (NY - 1) * DY_KM

times_s = np.arange(0, TOTAL_TIME_SECONDS + DT_SECONDS, DT_SECONDS)
times_min = times_s / 60.0
nt = len(times_s)

theta = np.deg2rad(WAVE_ANGLE_DEG)

k = 2.0 * np.pi / WAVE_LAMBDA_KM


# ==================================================
# Create time series
# ==================================================
rng = np.random.default_rng(RANDOM_SEED)

wave_cube = np.zeros((nt, NY, NX), dtype=np.float32)

for it, t in enumerate(times_s):
    displacement_km = SPEED_KMS * t

    # Moving wave packet center
    x_center_t = x0 + displacement_km * np.cos(theta)
    y_center_t = y0 + displacement_km * np.sin(theta)

    # Moving envelope
    envelope_t = np.exp(
        -0.5 * (
            ((x2d - x_center_t) / WAVE_SIGMA_X_KM) ** 2
            + ((y2d - y_center_t) / WAVE_SIGMA_Y_KM) ** 2
        )
    )

    # Directional coordinate measured from the original center
    s_km = (x2d - x0) * np.cos(theta) + (y2d - y0) * np.sin(theta)

    # Propagating phase
    phase = k * (s_km - displacement_km)

    field = WAVE_AMP * envelope_t * np.cos(phase)
    field += rng.normal(0.0, NOISE_STD, size=field.shape)

    wave_cube[it] = field.astype(np.float32)


vmax = np.nanpercentile(np.abs(wave_cube), 99)
vmin = -vmax


# ==================================================
# Animation
# ==================================================
fig, ax = plt.subplots(figsize=(7, 6))

im = ax.imshow(
    wave_cube[0],
    origin="upper",
    cmap=CMAP,
    vmin=vmin,
    vmax=vmax,
    extent=[x_km.min(), x_km.max(), y_km.max(), y_km.min()],
)

cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label("Synthetic wave amplitude")

title = ax.set_title("")
ax.set_xlabel("E-W distance (km)")
ax.set_ylabel("N-S distance (km)")
ax.set_aspect("equal")


def update(frame):
    im.set_data(wave_cube[frame])
    title.set_text(
        f"Moving Synthetic Wave Packet | "
        f"t = {times_min[frame]:.1f} min | "
        f"speed = {SPEED_MS:.0f} m/s"
    )
    return im, title


ani = FuncAnimation(
    fig,
    update,
    frames=nt,
    interval=1000 / FPS,
    blit=False,
)

out_mp4 = OUT_DIR / "moving_wave_packet_animation.mp4"

writer = FFMpegWriter(fps=FPS, bitrate=3000)
ani.save(out_mp4, writer=writer, dpi=DPI)

plt.close(fig)


# ==================================================
# Save first frame with directional keogram cuts
# ==================================================
fig_first, ax_first = plt.subplots(figsize=(7, 6))

im_first = ax_first.imshow(
    wave_cube[0],
    origin="upper",
    cmap=CMAP,
    vmin=vmin,
    vmax=vmax,
    extent=[x_km.min(), x_km.max(), y_km.max(), y_km.min()],
)

cbar_first = fig_first.colorbar(im_first, ax=ax_first, fraction=0.046, pad=0.04)
cbar_first.set_label("Synthetic wave amplitude")

cx_km = x0
cy_km = y0
half_length_km = 0.5 * min((NX - 1) * DX_KM, (NY - 1) * DY_KM)

for angle_deg in KEOGRAM_ANGLES_DEG:
    theta_cut = np.deg2rad(angle_deg)

    dx_line = half_length_km * np.cos(theta_cut)
    dy_line = half_length_km * np.sin(theta_cut)

    x_line = [cx_km - dx_line, cx_km + dx_line]
    y_line = [cy_km - dy_line, cy_km + dy_line]

    ax_first.plot(
        x_line,
        y_line,
        linewidth=1.2,
        alpha=0.8,
        label=f"{angle_deg:.0f}°",
    )

ax_first.set_title("First Frame with Directional Keogram Cuts")
ax_first.set_xlabel("E-W distance (km)")
ax_first.set_ylabel("N-S distance (km)")
ax_first.set_aspect("equal")

ax_first.legend(
    loc="upper right",
    fontsize=7,
    ncol=2,
    framealpha=0.8,
)

plt.tight_layout()

out_first = OUT_DIR / "first_frame_with_keogram_cuts.png"
fig_first.savefig(out_first, dpi=DPI, bbox_inches="tight")
plt.close(fig_first)


# ==================================================
# Directional keogram helper
# ==================================================
def make_directional_keogram(wave_cube, angle_deg, width_pixels=3):
    """
    angle_deg:
        0 deg  = E-W slice
        90 deg = N-S slice
    """

    theta = np.deg2rad(angle_deg)

    ux = np.cos(theta)
    uy = np.sin(theta)

    px = -np.sin(theta)
    py = np.cos(theta)

    cx = (NX - 1) / 2
    cy = (NY - 1) / 2

    half_length_pix = min(NX, NY) // 2
    distances_pix = np.arange(-half_length_pix, half_length_pix + 1)

    offsets = np.arange(width_pixels) - width_pixels // 2

    keogram = np.full((nt, len(distances_pix)), np.nan, dtype=float)

    for it in range(nt):
        sampled_slices = []

        for off in offsets:
            x_sample = cx + distances_pix * ux + off * px
            y_sample = cy + distances_pix * uy + off * py

            valid = (
                (x_sample >= 0) & (x_sample <= NX - 1) &
                (y_sample >= 0) & (y_sample <= NY - 1)
            )

            values = np.full(len(distances_pix), np.nan)

            values[valid] = map_coordinates(
                wave_cube[it],
                [y_sample[valid], x_sample[valid]],
                order=1,
                mode="nearest",
            )

            sampled_slices.append(values)

        keogram[it] = np.nanmean(sampled_slices, axis=0)

    distance_km = distances_pix * DX_KM

    return keogram, distance_km


# ==================================================
# Create and save directional keograms
# ==================================================
saved_keograms = []

for angle_deg in KEOGRAM_ANGLES_DEG:
    keogram, distance_km = make_directional_keogram(
        wave_cube,
        angle_deg,
        width_pixels=KEOGRAM_WIDTH_PIXELS,
    )

    fig_k, ax_k = plt.subplots(figsize=(9, 5))

    im_k = ax_k.imshow(
        keogram.T,
        origin="lower",
        aspect="auto",
        cmap=CMAP,
        vmin=vmin,
        vmax=vmax,
        extent=[
            times_min.min(),
            times_min.max(),
            distance_km.min(),
            distance_km.max(),
        ],
    )

    cbar_k = fig_k.colorbar(im_k, ax=ax_k, fraction=0.046, pad=0.04)
    cbar_k.set_label("Synthetic wave amplitude")

    ax_k.set_title(f"Directional Keogram: {angle_deg:.0f}°")
    ax_k.set_xlabel("Time (min)")
    ax_k.set_ylabel("Distance along slice (km)")

    plt.tight_layout()

    out_png = OUT_DIR / f"keogram_angle_{angle_deg:03.0f}.png"
    out_npy = OUT_DIR / f"keogram_angle_{angle_deg:03.0f}.npy"

    fig_k.savefig(out_png, dpi=DPI, bbox_inches="tight")
    plt.close(fig_k)

    np.save(out_npy, keogram)

    saved_keograms.append(out_png)


np.save(OUT_DIR / "times_s.npy", times_s)
np.save(OUT_DIR / "x_km.npy", x_km)
np.save(OUT_DIR / "y_km.npy", y_km)
np.save(OUT_DIR / "keogram_angles_deg.npy", KEOGRAM_ANGLES_DEG)


# ==================================================
# Save final frame
# ==================================================
fig_last, ax_last = plt.subplots(figsize=(7, 6))

im_last = ax_last.imshow(
    wave_cube[-1],
    origin="upper",
    cmap=CMAP,
    vmin=vmin,
    vmax=vmax,
    extent=[x_km.min(), x_km.max(), y_km.max(), y_km.min()],
)

cbar_last = fig_last.colorbar(im_last, ax=ax_last, fraction=0.046, pad=0.04)
cbar_last.set_label("Synthetic wave amplitude")

ax_last.set_title("Final Frame: Moving Synthetic Wave Packet")
ax_last.set_xlabel("E-W distance (km)")
ax_last.set_ylabel("N-S distance (km)")
ax_last.set_aspect("equal")

plt.tight_layout()

out_last = OUT_DIR / "final_frame.png"
fig_last.savefig(out_last, dpi=DPI, bbox_inches="tight")
plt.close(fig_last)


# ==================================================
# Print outputs
# ==================================================
print("Saved:")
print(out_mp4)
print(out_first)
print(out_last)

print("\nDirectional keograms:")
for f in saved_keograms:
    print(f)

print(f"\nNumber of directional keograms: {len(KEOGRAM_ANGLES_DEG)}")

C:\Users\domin\AppData\Local\Temp\ipykernel_72640\379443189.py:262: RuntimeWarning: Mean of empty slice
  keogram[it] = np.nanmean(sampled_slices, axis=0)


Saved:
propagating_wave_outputs\moving_wave_packet_animation.mp4
propagating_wave_outputs\first_frame_with_keogram_cuts.png
propagating_wave_outputs\final_frame.png

Directional keograms:
propagating_wave_outputs\keogram_angle_000.png
propagating_wave_outputs\keogram_angle_015.png
propagating_wave_outputs\keogram_angle_030.png
propagating_wave_outputs\keogram_angle_045.png
propagating_wave_outputs\keogram_angle_060.png
propagating_wave_outputs\keogram_angle_075.png
propagating_wave_outputs\keogram_angle_090.png
propagating_wave_outputs\keogram_angle_105.png
propagating_wave_outputs\keogram_angle_120.png
propagating_wave_outputs\keogram_angle_135.png
propagating_wave_outputs\keogram_angle_150.png
propagating_wave_outputs\keogram_angle_165.png

Number of directional keograms: 12


## (old) Line plots

In [10]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path


# ==================================================
# Settings
# ==================================================
OUT_DIR = Path("propagating_wave_outputs")

KEOGRAM_FILE = OUT_DIR / "keogram_angle_015.npy"
TIMES_FILE = OUT_DIR / "times_s.npy"

DPI = 200
CMAP = "RdBu_r"


# ==================================================
# Read 15-degree keogram
# ==================================================
keogram = np.load(KEOGRAM_FILE)
times_s = np.load(TIMES_FILE)
times_min = times_s / 60.0

nt, nspace = keogram.shape

# Reconstruct distance axis used in the original script
distance_km = np.arange(-(nspace // 2), nspace // 2 + 1)

if len(distance_km) != nspace:
    distance_km = np.arange(nspace) - nspace // 2


# ==================================================
# Extract top row, bottom row, and left column
# ==================================================
# keogram shape: (time, distance)
#
# After plotting with keogram.T:
#   Top row     = maximum distance
#   Bottom row  = minimum distance
#   Left column = first time

bottom_row = keogram[:, 0]
top_row = keogram[:, -1]
left_column = keogram[0, :]


# ==================================================
# Plot keogram with selected row/column shown
# ==================================================
vmax = np.nanpercentile(np.abs(keogram), 99)
vmin = -vmax

fig, ax = plt.subplots(figsize=(9, 5))

im = ax.imshow(
    keogram.T,
    origin="lower",
    aspect="auto",
    cmap=CMAP,
    vmin=vmin,
    vmax=vmax,
    extent=[
        times_min.min(),
        times_min.max(),
        distance_km.min(),
        distance_km.max(),
    ],
)

cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label("Synthetic wave amplitude")

ax.axhline(
    distance_km[0],
    color="black",
    linewidth=2.0,
    label="Bottom row",
)

ax.axhline(
    distance_km[-1],
    color="red",
    linewidth=2.0,
    label="Top row",
)

ax.axvline(
    times_min[0],
    color="gray",
    linewidth=2.0,
    label="Left column",
)

ax.set_title("15° Directional Keogram with Top Row, Bottom Row, and Left Column")
ax.set_xlabel("Time (min)")
ax.set_ylabel("Distance along slice (km)")
ax.legend(loc="upper right")

plt.tight_layout()

out_keogram_plot = OUT_DIR / "keogram_015_with_top_bottom_left.png"
fig.savefig(out_keogram_plot, dpi=DPI, bbox_inches="tight")
plt.close(fig)


# ==================================================
# Plot top row
# ==================================================
fig_t, ax_t = plt.subplots(figsize=(8, 4))

ax_t.plot(times_min, top_row, linewidth=1.5)

ax_t.set_title("Top Row of 15° Keogram")
ax_t.set_xlabel("Time (min)")
ax_t.set_ylabel("Synthetic wave amplitude")
ax_t.grid(True, alpha=0.3)

plt.tight_layout()

out_top = OUT_DIR / "keogram_015_top_row.png"
fig_t.savefig(out_top, dpi=DPI, bbox_inches="tight")
plt.close(fig_t)


# ==================================================
# Plot bottom row
# ==================================================
fig_b, ax_b = plt.subplots(figsize=(8, 4))

ax_b.plot(times_min, bottom_row, linewidth=1.5)

ax_b.set_title("Bottom Row of 15° Keogram")
ax_b.set_xlabel("Time (min)")
ax_b.set_ylabel("Synthetic wave amplitude")
ax_b.grid(True, alpha=0.3)

plt.tight_layout()

out_bottom = OUT_DIR / "keogram_015_bottom_row.png"
fig_b.savefig(out_bottom, dpi=DPI, bbox_inches="tight")
plt.close(fig_b)


# ==================================================
# Plot left column
# ==================================================
fig_l, ax_l = plt.subplots(figsize=(5, 6))

ax_l.plot(left_column, distance_km, linewidth=1.5)

ax_l.set_title("Left Column of 15° Keogram")
ax_l.set_xlabel("Synthetic wave amplitude")
ax_l.set_ylabel("Distance along slice (km)")
ax_l.grid(True, alpha=0.3)

plt.tight_layout()

out_left = OUT_DIR / "keogram_015_left_column.png"
fig_l.savefig(out_left, dpi=DPI, bbox_inches="tight")
plt.close(fig_l)


# ==================================================
# Print outputs
# ==================================================
print("Saved:")
print(out_keogram_plot)
print(out_top)
print(out_bottom)
print(out_left)

print("\nKeogram shape:")
print(f"time x distance = {keogram.shape}")

print("\nExtracted arrays:")
print(f"top_row shape = {top_row.shape}")
print(f"bottom_row shape = {bottom_row.shape}")
print(f"left_column shape = {left_column.shape}")

Saved:
propagating_wave_outputs\keogram_015_with_top_bottom_left.png
propagating_wave_outputs\keogram_015_top_row.png
propagating_wave_outputs\keogram_015_bottom_row.png
propagating_wave_outputs\keogram_015_left_column.png

Keogram shape:
time x distance = (481, 251)

Extracted arrays:
top_row shape = (481,)
bottom_row shape = (481,)
left_column shape = (251,)


## Reading pkl file instead

In [29]:
import pickle
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, FFMpegWriter
from pathlib import Path
from scipy.ndimage import map_coordinates


# ==================================================
# Settings
# ==================================================
PKL_FILE = r"C:\Users\domin\Downloads\processed_data_full_03_25_2025.pkl"

OUT_DIR = Path("BLO_perturbation_keogram_outputs_new")
OUT_DIR.mkdir(parents=True, exist_ok=True)

DX_KM = 1.0
DY_KM = 1.0

DT_SECONDS = 120

FPS = 20
DPI = 200

CMAP = "RdBu_r"

# Make keograms every 5 degrees
KEOGRAM_ANGLES_DEG = np.arange(0, 176, 5)

# Only draw every 15 degrees on the first frame
PLOT_CUT_ANGLES_DEG = np.arange(0, 176, 15)

KEOGRAM_WIDTH_PIXELS = 7 # Average a wider strip, less noisy. 

# Keogram display limits
KEOGRAM_TIME_MIN_MAX = 300
KEOGRAM_DISTANCE_MIN = -250
KEOGRAM_DISTANCE_MAX = 250


# ==================================================
# Load PKL perturbation array
# ==================================================
with open(PKL_FILE, "rb") as file:
    data = pickle.load(file)

# PKL format:
# data[5] = ["Perturbation Images", perturbation_array]
# data[7] = ["Datetime", [date, start_time]]
wave_cube = np.asarray(data[5][1], dtype=np.float32)
date_start = data[7][1]

# Expected shape: time, y, x
if wave_cube.ndim != 3:
    raise ValueError(
        f"Expected perturbation array with shape (time, y, x), got {wave_cube.shape}"
    )

nt, NY, NX = wave_cube.shape

print("Perturbation array shape:", wave_cube.shape)
print("Date/start time:", date_start)


# ==================================================
# Coordinate grid
# ==================================================
x_km = np.arange(NX) * DX_KM
y_km = np.arange(NY) * DY_KM

times_s = np.arange(nt) * DT_SECONDS
times_min = times_s / 60.0


# ==================================================
# Color limits
# ==================================================
vmax = np.nanpercentile(np.abs(wave_cube), 99)
if not np.isfinite(vmax) or vmax == 0:
    vmax = 1.0

vmin = -vmax


# ==================================================
# Animation
# ==================================================
fig, ax = plt.subplots(figsize=(7, 6))

im = ax.imshow(
    wave_cube[0],
    origin="upper",
    cmap=CMAP,
    vmin=vmin,
    vmax=vmax,
    extent=[x_km.min(), x_km.max(), y_km.max(), y_km.min()],
)

cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label("Perturbation amplitude")

title = ax.set_title("")
ax.set_xlabel("E-W distance (km)")
ax.set_ylabel("N-S distance (km)")
ax.set_aspect("equal")


def update(frame):
    im.set_data(wave_cube[frame])
    title.set_text(
        f"BLO Perturbation Image | "
        f"Frame {frame}/{nt - 1} | "
        f"t = {times_min[frame]:.1f} min"
    )
    return im, title


ani = FuncAnimation(
    fig,
    update,
    frames=nt,
    interval=1000 / FPS,
    blit=False,
)

out_mp4 = OUT_DIR / "BLO_perturbation_animation.mp4"

writer = FFMpegWriter(fps=FPS, bitrate=3000)
ani.save(out_mp4, writer=writer, dpi=DPI)

plt.close(fig)


# ==================================================
# Save first frame with directional keogram cuts
# Only show every 15 degrees, but still create keograms every 5 degrees later
# ==================================================
fig_first, ax_first = plt.subplots(figsize=(7, 6))

im_first = ax_first.imshow(
    wave_cube[0],
    origin="upper",
    cmap=CMAP,
    vmin=vmin,
    vmax=vmax,
    extent=[x_km.min(), x_km.max(), y_km.max(), y_km.min()],
)

cbar_first = fig_first.colorbar(im_first, ax=ax_first, fraction=0.046, pad=0.04)
cbar_first.set_label("Perturbation amplitude")

cx_km = 0.5 * (NX - 1) * DX_KM
cy_km = 0.5 * (NY - 1) * DY_KM
half_length_km = 0.5 * min((NX - 1) * DX_KM, (NY - 1) * DY_KM)

for angle_deg in PLOT_CUT_ANGLES_DEG:
    theta_cut = np.deg2rad(angle_deg)

    dx_line = half_length_km * np.cos(theta_cut)
    dy_line = half_length_km * np.sin(theta_cut)

    x_line = [cx_km - dx_line, cx_km + dx_line]
    y_line = [cy_km - dy_line, cy_km + dy_line]

    ax_first.plot(
        x_line,
        y_line,
        linewidth=1.4,
        alpha=0.85,
        label=f"{angle_deg:.0f}°",
    )

ax_first.set_title("First Perturbation Frame with Directional Keogram Cuts")
ax_first.set_xlabel("E-W distance (km)")
ax_first.set_ylabel("N-S distance (km)")
ax_first.set_aspect("equal")

ax_first.legend(
    loc="upper right",
    fontsize=7,
    ncol=2,
    framealpha=0.8,
)

plt.tight_layout()

out_first = OUT_DIR / "first_perturbation_frame_with_keogram_cuts_every_15deg.png"
fig_first.savefig(out_first, dpi=DPI, bbox_inches="tight")
plt.close(fig_first)


# ==================================================
# Directional keogram helper
# ==================================================
def make_directional_keogram(wave_cube, angle_deg, width_pixels=3):
    """
    angle_deg:
        0 deg  = E-W slice
        90 deg = N-S slice
    """

    nt, NY, NX = wave_cube.shape

    theta = np.deg2rad(angle_deg)

    ux = np.cos(theta)
    uy = np.sin(theta)

    px = -np.sin(theta)
    py = np.cos(theta)

    cx = (NX - 1) / 2
    cy = (NY - 1) / 2

    half_length_pix = min(NX, NY) // 2
    distances_pix = np.arange(-half_length_pix, half_length_pix + 1)

    offsets = np.arange(width_pixels) - width_pixels // 2

    keogram = np.full((nt, len(distances_pix)), np.nan, dtype=float)

    for it in range(nt):
        sampled_slices = []

        for off in offsets:
            x_sample = cx + distances_pix * ux + off * px
            y_sample = cy + distances_pix * uy + off * py

            valid = (
                (x_sample >= 0) & (x_sample <= NX - 1) &
                (y_sample >= 0) & (y_sample <= NY - 1)
            )

            values = np.full(len(distances_pix), np.nan)

            values[valid] = map_coordinates(
                wave_cube[it],
                [y_sample[valid], x_sample[valid]],
                order=1,
                mode="nearest",
            )

            sampled_slices.append(values)

        keogram[it] = np.nanmean(sampled_slices, axis=0)

    distance_km = distances_pix * DX_KM

    return keogram, distance_km


# ==================================================
# Create and save directional keograms
# Make every 5 degrees
# Display only 0 to 300 min and -250 to 250 km
# Save full keogram array as .npy
# ==================================================
saved_keograms = []

for angle_deg in KEOGRAM_ANGLES_DEG:
    keogram, distance_km = make_directional_keogram(
        wave_cube,
        angle_deg,
        width_pixels=KEOGRAM_WIDTH_PIXELS,
    )

    fig_k, ax_k = plt.subplots(figsize=(9, 5))

    im_k = ax_k.imshow(
        keogram.T,
        origin="lower",
        aspect="auto",
        cmap=CMAP,
        vmin=vmin,
        vmax=vmax,
        extent=[
            times_min.min(),
            times_min.max(),
            distance_km.min(),
            distance_km.max(),
        ],
    )

    cbar_k = fig_k.colorbar(im_k, ax=ax_k, fraction=0.046, pad=0.04)
    cbar_k.set_label("Perturbation amplitude")

    ax_k.set_title(f"BLO Directional Keogram: {angle_deg:.0f}°")
    ax_k.set_xlabel("Time (min)")
    ax_k.set_ylabel("Distance along slice (km)")

    ax_k.set_xlim(0, KEOGRAM_TIME_MIN_MAX)
    ax_k.set_ylim(KEOGRAM_DISTANCE_MIN, KEOGRAM_DISTANCE_MAX)

    plt.tight_layout()

    out_png = OUT_DIR / f"BLO_keogram_angle_{angle_deg:03.0f}.png"
    out_npy = OUT_DIR / f"BLO_keogram_angle_{angle_deg:03.0f}.npy"

    fig_k.savefig(out_png, dpi=DPI, bbox_inches="tight")
    plt.close(fig_k)

    np.save(out_npy, keogram)

    saved_keograms.append(out_png)


# ==================================================
# Save arrays and metadata
# ==================================================
np.save(OUT_DIR / "times_s.npy", times_s)
np.save(OUT_DIR / "x_km.npy", x_km)
np.save(OUT_DIR / "y_km.npy", y_km)
np.save(OUT_DIR / "keogram_angles_deg.npy", KEOGRAM_ANGLES_DEG)
np.save(OUT_DIR / "first_frame_cut_angles_deg.npy", PLOT_CUT_ANGLES_DEG)

with open(OUT_DIR / "Date_and_Start_Time.txt", "w") as f:
    f.write(f"Date: {date_start[0]}\n")
    f.write(f"Start time: {date_start[1]}\n")


# ==================================================
# Save final frame
# ==================================================
fig_last, ax_last = plt.subplots(figsize=(7, 6))

im_last = ax_last.imshow(
    wave_cube[-1],
    origin="upper",
    cmap=CMAP,
    vmin=vmin,
    vmax=vmax,
    extent=[x_km.min(), x_km.max(), y_km.max(), y_km.min()],
)

cbar_last = fig_last.colorbar(im_last, ax=ax_last, fraction=0.046, pad=0.04)
cbar_last.set_label("Perturbation amplitude")

ax_last.set_title("Final BLO Perturbation Frame")
ax_last.set_xlabel("E-W distance (km)")
ax_last.set_ylabel("N-S distance (km)")
ax_last.set_aspect("equal")

plt.tight_layout()

out_last = OUT_DIR / "final_perturbation_frame.png"
fig_last.savefig(out_last, dpi=DPI, bbox_inches="tight")
plt.close(fig_last)


# ==================================================
# Print outputs
# ==================================================
print("Saved:")
print(out_mp4)
print(out_first)
print(out_last)

print("\nDirectional keograms:")
for f in saved_keograms:
    print(f)

print(f"\nNumber of directional keograms made: {len(KEOGRAM_ANGLES_DEG)}")
print(f"First-frame cut lines shown: every 15 degrees, {len(PLOT_CUT_ANGLES_DEG)} lines")
print("Keogram plot x-limit: 0 to 300 min")
print("Keogram plot y-limit: -250 to 250 km")
print("Output folder:", OUT_DIR)

Perturbation array shape: (280, 1000, 1000)
Date/start time: ['03_25_2025', '20:29:27']


C:\Users\domin\AppData\Local\Temp\ipykernel_16424\3937517638.py:245: RuntimeWarning: Mean of empty slice
  keogram[it] = np.nanmean(sampled_slices, axis=0)


Saved:
BLO_perturbation_keogram_outputs_new\BLO_perturbation_animation.mp4
BLO_perturbation_keogram_outputs_new\first_perturbation_frame_with_keogram_cuts_every_15deg.png
BLO_perturbation_keogram_outputs_new\final_perturbation_frame.png

Directional keograms:
BLO_perturbation_keogram_outputs_new\BLO_keogram_angle_000.png
BLO_perturbation_keogram_outputs_new\BLO_keogram_angle_005.png
BLO_perturbation_keogram_outputs_new\BLO_keogram_angle_010.png
BLO_perturbation_keogram_outputs_new\BLO_keogram_angle_015.png
BLO_perturbation_keogram_outputs_new\BLO_keogram_angle_020.png
BLO_perturbation_keogram_outputs_new\BLO_keogram_angle_025.png
BLO_perturbation_keogram_outputs_new\BLO_keogram_angle_030.png
BLO_perturbation_keogram_outputs_new\BLO_keogram_angle_035.png
BLO_perturbation_keogram_outputs_new\BLO_keogram_angle_040.png
BLO_perturbation_keogram_outputs_new\BLO_keogram_angle_045.png
BLO_perturbation_keogram_outputs_new\BLO_keogram_angle_050.png
BLO_perturbation_keogram_outputs_new\BLO_keogra

## Read pkl, calculate slope

In [32]:
import pickle
import csv
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, FFMpegWriter
from pathlib import Path
from scipy.ndimage import map_coordinates


# ==================================================
# Settings
# ==================================================
PKL_FILE = r"C:\Users\domin\Downloads\processed_data_full_03_25_2025.pkl"

OUT_DIR = Path("BLO_perturbation_keogram_outputs_new")
OUT_DIR.mkdir(parents=True, exist_ok=True)

DX_KM = 1.0
DY_KM = 1.0

DT_SECONDS = 120

FPS = 20
DPI = 200

CMAP = "RdBu_r"

# Make keograms every 5 degrees
KEOGRAM_ANGLES_DEG = np.arange(0, 176, 5)

# Only draw every 15 degrees on the first frame
PLOT_CUT_ANGLES_DEG = np.arange(0, 176, 15)

# Average a wider strip, less noisy
KEOGRAM_WIDTH_PIXELS = 7

# Keogram display limits
KEOGRAM_TIME_MIN_MAX = 300
KEOGRAM_DISTANCE_MIN = -250
KEOGRAM_DISTANCE_MAX = 250

# 2D FFT settings
SAVE_FFT_IMAGES = True
FFT_EXCLUDE_RADIUS_PIXELS = 5


# ==================================================
# Load PKL perturbation array
# ==================================================
with open(PKL_FILE, "rb") as file:
    data = pickle.load(file)

# PKL format:
# data[5] = ["Perturbation Images", perturbation_array]
# data[7] = ["Datetime", [date, start_time]]
wave_cube = np.asarray(data[5][1], dtype=np.float32)
date_start = data[7][1]

# Expected shape: time, y, x
if wave_cube.ndim != 3:
    raise ValueError(
        f"Expected perturbation array with shape (time, y, x), got {wave_cube.shape}"
    )

nt, NY, NX = wave_cube.shape

print("Perturbation array shape:", wave_cube.shape)
print("Date/start time:", date_start)


# ==================================================
# Coordinate grid
# ==================================================
x_km = np.arange(NX) * DX_KM
y_km = np.arange(NY) * DY_KM

times_s = np.arange(nt) * DT_SECONDS
times_min = times_s / 60.0


# ==================================================
# Color limits
# ==================================================
vmax = np.nanpercentile(np.abs(wave_cube), 99)
if not np.isfinite(vmax) or vmax == 0:
    vmax = 1.0

vmin = -vmax


# ==================================================
# Animation
# ==================================================
fig, ax = plt.subplots(figsize=(7, 6))

im = ax.imshow(
    wave_cube[0],
    origin="upper",
    cmap=CMAP,
    vmin=vmin,
    vmax=vmax,
    extent=[x_km.min(), x_km.max(), y_km.max(), y_km.min()],
)

cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label("Perturbation amplitude")

title = ax.set_title("")
ax.set_xlabel("E-W distance (km)")
ax.set_ylabel("N-S distance (km)")
ax.set_aspect("equal")


def update(frame):
    im.set_data(wave_cube[frame])
    title.set_text(
        f"BLO Perturbation Image | "
        f"Frame {frame}/{nt - 1} | "
        f"t = {times_min[frame]:.1f} min"
    )
    return im, title


ani = FuncAnimation(
    fig,
    update,
    frames=nt,
    interval=1000 / FPS,
    blit=False,
)

out_mp4 = OUT_DIR / "BLO_perturbation_animation.mp4"

writer = FFMpegWriter(fps=FPS, bitrate=3000)
ani.save(out_mp4, writer=writer, dpi=DPI)

plt.close(fig)


# ==================================================
# Save first frame with directional keogram cuts
# Only show every 15 degrees, but still create keograms every 5 degrees later
# ==================================================
fig_first, ax_first = plt.subplots(figsize=(7, 6))

im_first = ax_first.imshow(
    wave_cube[0],
    origin="upper",
    cmap=CMAP,
    vmin=vmin,
    vmax=vmax,
    extent=[x_km.min(), x_km.max(), y_km.max(), y_km.min()],
)

cbar_first = fig_first.colorbar(im_first, ax=ax_first, fraction=0.046, pad=0.04)
cbar_first.set_label("Perturbation amplitude")

cx_km = 0.5 * (NX - 1) * DX_KM
cy_km = 0.5 * (NY - 1) * DY_KM
half_length_km = 0.5 * min((NX - 1) * DX_KM, (NY - 1) * DY_KM)

for angle_deg in PLOT_CUT_ANGLES_DEG:
    theta_cut = np.deg2rad(angle_deg)

    dx_line = half_length_km * np.cos(theta_cut)
    dy_line = half_length_km * np.sin(theta_cut)

    x_line = [cx_km - dx_line, cx_km + dx_line]
    y_line = [cy_km - dy_line, cy_km + dy_line]

    ax_first.plot(
        x_line,
        y_line,
        linewidth=1.4,
        alpha=0.85,
        label=f"{angle_deg:.0f}°",
    )

ax_first.set_title("First Perturbation Frame with Directional Keogram Cuts")
ax_first.set_xlabel("E-W distance (km)")
ax_first.set_ylabel("N-S distance (km)")
ax_first.set_aspect("equal")

ax_first.legend(
    loc="upper right",
    fontsize=7,
    ncol=2,
    framealpha=0.8,
)

plt.tight_layout()

out_first = OUT_DIR / "first_perturbation_frame_with_keogram_cuts_every_15deg.png"
fig_first.savefig(out_first, dpi=DPI, bbox_inches="tight")
plt.close(fig_first)


# ==================================================
# Directional keogram helper
# ==================================================
def make_directional_keogram(wave_cube, angle_deg, width_pixels=3):
    """
    angle_deg:
        0 deg  = E-W slice
        90 deg = N-S slice
    """

    nt, NY, NX = wave_cube.shape

    theta = np.deg2rad(angle_deg)

    ux = np.cos(theta)
    uy = np.sin(theta)

    px = -np.sin(theta)
    py = np.cos(theta)

    cx = (NX - 1) / 2
    cy = (NY - 1) / 2

    half_length_pix = min(NX, NY) // 2
    distances_pix = np.arange(-half_length_pix, half_length_pix + 1)

    offsets = np.arange(width_pixels) - width_pixels // 2

    keogram = np.full((nt, len(distances_pix)), np.nan, dtype=float)

    for it in range(nt):
        sampled_slices = []

        for off in offsets:
            x_sample = cx + distances_pix * ux + off * px
            y_sample = cy + distances_pix * uy + off * py

            valid = (
                (x_sample >= 0) & (x_sample <= NX - 1) &
                (y_sample >= 0) & (y_sample <= NY - 1)
            )

            values = np.full(len(distances_pix), np.nan)

            values[valid] = map_coordinates(
                wave_cube[it],
                [y_sample[valid], x_sample[valid]],
                order=1,
                mode="nearest",
            )

            sampled_slices.append(values)

        keogram[it] = np.nanmean(sampled_slices, axis=0)

    distance_km = distances_pix * DX_KM

    return keogram, distance_km


# ==================================================
# 2D FFT helper for keogram slope
# ==================================================
def estimate_keogram_fft_slope(keogram, exclude_radius_pixels=5):
    """
    Estimate dominant keogram orientation using 2D FFT.

    keogram shape:
        axis 0 = time pixels
        axis 1 = distance pixels

    Returns:
        slope_distance_pixel_per_time_pixel

    Interpretation:
        slope > 0 means feature tilts upward to the right
        slope < 0 means feature tilts downward to the right
    """

    Z = np.asarray(keogram, dtype=float)

    mean_val = np.nanmean(Z)
    Z = np.where(np.isfinite(Z), Z, mean_val)

    # Remove mean
    Z = Z - np.mean(Z)

    nt, nd = Z.shape

    # Apply 2D Hann window to reduce edge artifacts
    win_t = np.hanning(nt)
    win_d = np.hanning(nd)
    window = win_t[:, None] * win_d[None, :]
    Z_win = Z * window

    # 2D FFT
    F = np.fft.fftshift(np.fft.fft2(Z_win))
    power = np.abs(F) ** 2

    # Frequency axes in cycles/pixel
    freq_t = np.fft.fftshift(np.fft.fftfreq(nt))
    freq_d = np.fft.fftshift(np.fft.fftfreq(nd))

    center_t = nt // 2
    center_d = nd // 2

    # Exclude DC and very low frequencies
    yy, xx = np.indices(power.shape)
    rr = np.sqrt((yy - center_t) ** 2 + (xx - center_d) ** 2)

    power_masked = power.copy()
    power_masked[rr <= exclude_radius_pixels] = 0

    # Find strongest FFT peak
    peak_t_idx, peak_d_idx = np.unravel_index(
        np.argmax(power_masked),
        power_masked.shape,
    )

    ft = freq_t[peak_t_idx]
    fd = freq_d[peak_d_idx]

    peak_power = power_masked[peak_t_idx, peak_d_idx]

    # Constant phase lines:
    # ft*time_pixel + fd*distance_pixel = constant
    #
    # distance_pixel / time_pixel = -ft / fd
    if np.isclose(fd, 0):
        slope_distance_per_time_pixel = np.inf
        orientation_angle_deg = 90.0
    else:
        slope_distance_per_time_pixel = -ft / fd
        orientation_angle_deg = np.rad2deg(
            np.arctan2(slope_distance_per_time_pixel, 1.0)
        )

    return {
        "peak_t_idx": peak_t_idx,
        "peak_d_idx": peak_d_idx,
        "freq_time_cycles_per_pixel": ft,
        "freq_distance_cycles_per_pixel": fd,
        "slope_distance_pixel_per_time_pixel": slope_distance_per_time_pixel,
        "orientation_angle_deg": orientation_angle_deg,
        "peak_power": peak_power,
        "power_spectrum": power,
        "power_masked": power_masked,
        "freq_t": freq_t,
        "freq_d": freq_d,
    }


# ==================================================
# Create and save directional keograms
# Also estimate dominant keogram slope using 2D FFT
# ==================================================
saved_keograms = []
saved_fft_images = []
fft_summary_rows = []

for angle_deg in KEOGRAM_ANGLES_DEG:
    keogram, distance_km = make_directional_keogram(
        wave_cube,
        angle_deg,
        width_pixels=KEOGRAM_WIDTH_PIXELS,
    )

    # ------------------------------
    # Save keogram plot
    # ------------------------------
    fig_k, ax_k = plt.subplots(figsize=(9, 5))

    im_k = ax_k.imshow(
        keogram.T,
        origin="lower",
        aspect="auto",
        cmap=CMAP,
        vmin=vmin,
        vmax=vmax,
        extent=[
            times_min.min(),
            times_min.max(),
            distance_km.min(),
            distance_km.max(),
        ],
    )

    cbar_k = fig_k.colorbar(im_k, ax=ax_k, fraction=0.046, pad=0.04)
    cbar_k.set_label("Perturbation amplitude")

    ax_k.set_title(f"BLO Directional Keogram: {angle_deg:.0f}°")
    ax_k.set_xlabel("Time (min)")
    ax_k.set_ylabel("Distance along slice (km)")

    ax_k.set_xlim(0, KEOGRAM_TIME_MIN_MAX)
    ax_k.set_ylim(KEOGRAM_DISTANCE_MIN, KEOGRAM_DISTANCE_MAX)

    plt.tight_layout()

    out_png = OUT_DIR / f"BLO_keogram_angle_{angle_deg:03.0f}.png"
    out_npy = OUT_DIR / f"BLO_keogram_angle_{angle_deg:03.0f}.npy"

    fig_k.savefig(out_png, dpi=DPI, bbox_inches="tight")
    plt.close(fig_k)

    np.save(out_npy, keogram)
    saved_keograms.append(out_png)

    # ------------------------------
    # 2D FFT slope estimate
    # ------------------------------
    fft_result = estimate_keogram_fft_slope(
        keogram,
        exclude_radius_pixels=FFT_EXCLUDE_RADIUS_PIXELS,
    )

    fft_summary_rows.append({
        "angle_deg": float(angle_deg),
        "peak_t_idx": int(fft_result["peak_t_idx"]),
        "peak_d_idx": int(fft_result["peak_d_idx"]),
        "freq_time_cycles_per_pixel": float(fft_result["freq_time_cycles_per_pixel"]),
        "freq_distance_cycles_per_pixel": float(fft_result["freq_distance_cycles_per_pixel"]),
        "slope_distance_pixel_per_time_pixel": float(
            fft_result["slope_distance_pixel_per_time_pixel"]
        ),
        "orientation_angle_deg": float(fft_result["orientation_angle_deg"]),
        "peak_power": float(fft_result["peak_power"]),
    })

    # ------------------------------
    # Save FFT image
    # ------------------------------
    if SAVE_FFT_IMAGES:
        power_plot = np.log10(fft_result["power_spectrum"] + 1)

        fig_fft, ax_fft = plt.subplots(figsize=(6, 5))

        im_fft = ax_fft.imshow(
            power_plot,
            origin="lower",
            aspect="auto",
            cmap="gray",
        )

        ax_fft.plot(
            fft_result["peak_d_idx"],
            fft_result["peak_t_idx"],
            "ro",
            markersize=5,
        )

        ax_fft.set_title(
            f"2D FFT Power: {angle_deg:.0f}°\n"
            f"slope = {fft_result['slope_distance_pixel_per_time_pixel']:.3f} pix/pix"
        )
        ax_fft.set_xlabel("Distance-frequency index")
        ax_fft.set_ylabel("Time-frequency index")

        cbar_fft = fig_fft.colorbar(im_fft, ax=ax_fft, fraction=0.046, pad=0.04)
        cbar_fft.set_label("log10(power + 1)")

        plt.tight_layout()

        out_fft_png = OUT_DIR / f"BLO_keogram_angle_{angle_deg:03.0f}_fft.png"
        fig_fft.savefig(out_fft_png, dpi=DPI, bbox_inches="tight")
        plt.close(fig_fft)

        saved_fft_images.append(out_fft_png)


# ==================================================
# Save FFT slope summary CSV
# ==================================================
fft_csv = OUT_DIR / "keogram_fft_slope_summary.csv"

with open(fft_csv, "w", newline="") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=[
            "angle_deg",
            "peak_t_idx",
            "peak_d_idx",
            "freq_time_cycles_per_pixel",
            "freq_distance_cycles_per_pixel",
            "slope_distance_pixel_per_time_pixel",
            "orientation_angle_deg",
            "peak_power",
        ],
    )
    writer.writeheader()
    writer.writerows(fft_summary_rows)


# ==================================================
# Save arrays and metadata
# ==================================================
np.save(OUT_DIR / "times_s.npy", times_s)
np.save(OUT_DIR / "x_km.npy", x_km)
np.save(OUT_DIR / "y_km.npy", y_km)
np.save(OUT_DIR / "keogram_angles_deg.npy", KEOGRAM_ANGLES_DEG)
np.save(OUT_DIR / "first_frame_cut_angles_deg.npy", PLOT_CUT_ANGLES_DEG)

with open(OUT_DIR / "Date_and_Start_Time.txt", "w") as f:
    f.write(f"Date: {date_start[0]}\n")
    f.write(f"Start time: {date_start[1]}\n")


# ==================================================
# Save final frame
# ==================================================
fig_last, ax_last = plt.subplots(figsize=(7, 6))

im_last = ax_last.imshow(
    wave_cube[-1],
    origin="upper",
    cmap=CMAP,
    vmin=vmin,
    vmax=vmax,
    extent=[x_km.min(), x_km.max(), y_km.max(), y_km.min()],
)

cbar_last = fig_last.colorbar(im_last, ax=ax_last, fraction=0.046, pad=0.04)
cbar_last.set_label("Perturbation amplitude")

ax_last.set_title("Final BLO Perturbation Frame")
ax_last.set_xlabel("E-W distance (km)")
ax_last.set_ylabel("N-S distance (km)")
ax_last.set_aspect("equal")

plt.tight_layout()

out_last = OUT_DIR / "final_perturbation_frame.png"
fig_last.savefig(out_last, dpi=DPI, bbox_inches="tight")
plt.close(fig_last)


# ==================================================
# Print outputs
# ==================================================
print("Saved:")
print(out_mp4)
print(out_first)
print(out_last)

print("\nDirectional keograms:")
for f in saved_keograms:
    print(f)

print("\nFFT slope summary saved:")
print(fft_csv)

if SAVE_FFT_IMAGES:
    print("\nFFT images:")
    for f in saved_fft_images:
        print(f)

print(f"\nNumber of directional keograms made: {len(KEOGRAM_ANGLES_DEG)}")
print(f"First-frame cut lines shown: every 15 degrees, {len(PLOT_CUT_ANGLES_DEG)} lines")
print("Keogram plot x-limit: 0 to 300 min")
print("Keogram plot y-limit: -250 to 250 km")
print("Output folder:", OUT_DIR)

Perturbation array shape: (280, 1000, 1000)
Date/start time: ['03_25_2025', '20:29:27']


C:\Users\domin\AppData\Local\Temp\ipykernel_16424\3796269613.py:251: RuntimeWarning: Mean of empty slice
  keogram[it] = np.nanmean(sampled_slices, axis=0)


Saved:
BLO_perturbation_keogram_outputs_new\BLO_perturbation_animation.mp4
BLO_perturbation_keogram_outputs_new\first_perturbation_frame_with_keogram_cuts_every_15deg.png
BLO_perturbation_keogram_outputs_new\final_perturbation_frame.png

Directional keograms:
BLO_perturbation_keogram_outputs_new\BLO_keogram_angle_000.png
BLO_perturbation_keogram_outputs_new\BLO_keogram_angle_005.png
BLO_perturbation_keogram_outputs_new\BLO_keogram_angle_010.png
BLO_perturbation_keogram_outputs_new\BLO_keogram_angle_015.png
BLO_perturbation_keogram_outputs_new\BLO_keogram_angle_020.png
BLO_perturbation_keogram_outputs_new\BLO_keogram_angle_025.png
BLO_perturbation_keogram_outputs_new\BLO_keogram_angle_030.png
BLO_perturbation_keogram_outputs_new\BLO_keogram_angle_035.png
BLO_perturbation_keogram_outputs_new\BLO_keogram_angle_040.png
BLO_perturbation_keogram_outputs_new\BLO_keogram_angle_045.png
BLO_perturbation_keogram_outputs_new\BLO_keogram_angle_050.png
BLO_perturbation_keogram_outputs_new\BLO_keogra

Date: 03_25_2025
Start time: 20:29:27

## Line plots for BLO keogram

In [15]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path


# ==================================================
# Settings
# ==================================================
OUT_DIR = Path("BLO_perturbation_keogram_outputs_new")

ANGLE_DEG = 40

KEOGRAM_FILE = OUT_DIR / f"BLO_keogram_angle_{ANGLE_DEG:03d}.npy"
TIMES_FILE = OUT_DIR / "times_s.npy"

DPI = 200
CMAP = "RdBu_r"

TIMES_TO_PLOT = [0, 100, 150, 200]

KEOGRAM_TIME_MIN_MAX = 300
KEOGRAM_DISTANCE_MIN = -250
KEOGRAM_DISTANCE_MAX = 250

ROWS_TO_PLOT_KM = [-250, 0, 250]


# ==================================================
# Read keogram
# ==================================================
keogram = np.load(KEOGRAM_FILE)
times_s = np.load(TIMES_FILE)
times_min = times_s / 60.0

nt, nspace = keogram.shape

distance_km = np.arange(-(nspace // 2), nspace // 2 + 1)

if len(distance_km) != nspace:
    distance_km = np.arange(nspace) - nspace // 2


# ==================================================
# Select rows at -250 km, 0 km, and 250 km
# ==================================================
idx_neg250 = np.argmin(np.abs(distance_km - (-250)))
idx_0km = np.argmin(np.abs(distance_km - 0))
idx_pos250 = np.argmin(np.abs(distance_km - 250))

row_neg250 = keogram[:, idx_neg250]
row_0km = keogram[:, idx_0km]
row_pos250 = keogram[:, idx_pos250]


# ==================================================
# Select time columns
# ==================================================
column_indices = []
columns = []

for t in TIMES_TO_PLOT:
    idx = np.argmin(np.abs(times_min - t))
    column_indices.append(idx)
    columns.append(keogram[idx, :])


print("Selected rows:")
print(f"-250 km row index: {idx_neg250}, distance = {distance_km[idx_neg250]} km")
print(f"   0 km row index: {idx_0km}, distance = {distance_km[idx_0km]} km")
print(f" 250 km row index: {idx_pos250}, distance = {distance_km[idx_pos250]} km")

print("\nSelected columns:")
for t, idx in zip(TIMES_TO_PLOT, column_indices):
    print(f"{t:6.1f} min -> frame {idx}, actual time = {times_min[idx]:.2f} min")


# ==================================================
# Plot keogram with selected rows and time columns
# ==================================================
vmax = np.nanpercentile(np.abs(keogram), 99)

if not np.isfinite(vmax) or vmax == 0:
    vmax = 1.0

vmin = -vmax

fig, ax = plt.subplots(figsize=(9, 5))

im = ax.imshow(
    keogram.T,
    origin="lower",
    aspect="auto",
    cmap=CMAP,
    vmin=vmin,
    vmax=vmax,
    extent=[
        times_min.min(),
        times_min.max(),
        distance_km.min(),
        distance_km.max(),
    ],
)

cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label("Perturbation amplitude")

ax.axhline(
    distance_km[idx_neg250],
    color="black",
    linewidth=2.0,
    label="-250 km",
)

ax.axhline(
    distance_km[idx_0km],
    color="gray",
    linewidth=2.0,
    linestyle="--",
    label="0 km",
)

ax.axhline(
    distance_km[idx_pos250],
    color="red",
    linewidth=2.0,
    label="250 km",
)

time_colors = ["black", "blue", "green", "purple"]

for t, idx, color in zip(TIMES_TO_PLOT, column_indices, time_colors):
    ax.axvline(
        times_min[idx],
        color=color,
        linewidth=1.8,
        linestyle=":",
        label=f"{t:.0f} min",
    )

ax.set_xlim(0, KEOGRAM_TIME_MIN_MAX)
ax.set_ylim(KEOGRAM_DISTANCE_MIN, KEOGRAM_DISTANCE_MAX)

ax.set_title(f"{ANGLE_DEG}° BLO Directional Keogram with Selected Rows and Time Columns")
ax.set_xlabel("Time (min)")
ax.set_ylabel("Distance along slice (km)")
ax.legend(loc="upper right", fontsize=8)

plt.tight_layout()

out_keogram_plot = OUT_DIR / f"BLO_keogram_{ANGLE_DEG:03d}_with_selected_rows_and_columns.png"
fig.savefig(out_keogram_plot, dpi=DPI, bbox_inches="tight")
plt.close(fig)


# ==================================================
# Plot -250 km row
# ==================================================
fig_neg250, ax_neg250 = plt.subplots(figsize=(8, 4))

ax_neg250.plot(times_min, row_neg250, linewidth=1.5)

ax_neg250.set_xlim(0, KEOGRAM_TIME_MIN_MAX)

ax_neg250.set_title(f"-250 km Row of {ANGLE_DEG}° BLO Keogram")
ax_neg250.set_xlabel("Time (min)")
ax_neg250.set_ylabel("Perturbation amplitude")
ax_neg250.grid(True, alpha=0.3)

plt.tight_layout()

out_neg250 = OUT_DIR / f"BLO_keogram_{ANGLE_DEG:03d}_minus250km_row.png"
fig_neg250.savefig(out_neg250, dpi=DPI, bbox_inches="tight")
plt.close(fig_neg250)


# ==================================================
# Plot 0-km row
# ==================================================
fig_0, ax_0 = plt.subplots(figsize=(8, 4))

ax_0.plot(times_min, row_0km, linewidth=1.5)

ax_0.set_xlim(0, KEOGRAM_TIME_MIN_MAX)

ax_0.set_title(f"0-km Row of {ANGLE_DEG}° BLO Keogram")
ax_0.set_xlabel("Time (min)")
ax_0.set_ylabel("Perturbation amplitude")
ax_0.grid(True, alpha=0.3)

plt.tight_layout()

out_0 = OUT_DIR / f"BLO_keogram_{ANGLE_DEG:03d}_0km_row.png"
fig_0.savefig(out_0, dpi=DPI, bbox_inches="tight")
plt.close(fig_0)


# ==================================================
# Plot 250 km row
# ==================================================
fig_pos250, ax_pos250 = plt.subplots(figsize=(8, 4))

ax_pos250.plot(times_min, row_pos250, linewidth=1.5)

ax_pos250.set_xlim(0, KEOGRAM_TIME_MIN_MAX)

ax_pos250.set_title(f"250 km Row of {ANGLE_DEG}° BLO Keogram")
ax_pos250.set_xlabel("Time (min)")
ax_pos250.set_ylabel("Perturbation amplitude")
ax_pos250.grid(True, alpha=0.3)

plt.tight_layout()

out_pos250 = OUT_DIR / f"BLO_keogram_{ANGLE_DEG:03d}_250km_row.png"
fig_pos250.savefig(out_pos250, dpi=DPI, bbox_inches="tight")
plt.close(fig_pos250)


# ==================================================
# Plot selected time columns/profiles
# ==================================================
fig_c, ax_c = plt.subplots(figsize=(6, 7))

for t, idx, col, color in zip(TIMES_TO_PLOT, column_indices, columns, time_colors):
    ax_c.plot(
        col,
        distance_km,
        linewidth=1.5,
        color=color,
        label=f"{t:.0f} min",
    )

ax_c.axhline(
    distance_km[idx_neg250],
    color="black",
    linestyle="--",
    linewidth=1.2,
    label="-250 km",
)

ax_c.axhline(
    distance_km[idx_0km],
    color="gray",
    linestyle="--",
    linewidth=1.2,
    label="0 km",
)

ax_c.axhline(
    distance_km[idx_pos250],
    color="red",
    linestyle="--",
    linewidth=1.2,
    label="250 km",
)

ax_c.set_ylim(KEOGRAM_DISTANCE_MIN, KEOGRAM_DISTANCE_MAX)

ax_c.set_title(f"Profiles Along {ANGLE_DEG}° Slice at Selected Times")
ax_c.set_xlabel("Perturbation amplitude")
ax_c.set_ylabel("Distance along slice (km)")
ax_c.grid(True, alpha=0.3)
ax_c.legend(loc="best", fontsize=8)

plt.tight_layout()

time_label = "_".join([f"{int(t)}" for t in TIMES_TO_PLOT])
out_columns = OUT_DIR / f"BLO_keogram_{ANGLE_DEG:03d}_profiles_{time_label}min.png"
fig_c.savefig(out_columns, dpi=DPI, bbox_inches="tight")
plt.close(fig_c)


# ==================================================
# Print outputs
# ==================================================
print("\nSaved:")
print(out_keogram_plot)
print(out_neg250)
print(out_0)
print(out_pos250)
print(out_columns)

print("\nKeogram shape:")
print(f"time x distance = {keogram.shape}")

print("\nExtracted row arrays:")
print(f"row_neg250 shape = {row_neg250.shape}")
print(f"row_0km shape = {row_0km.shape}")
print(f"row_pos250 shape = {row_pos250.shape}")

print("\nExtracted column arrays:")
for t, col in zip(TIMES_TO_PLOT, columns):
    print(f"{t:.0f} min column shape = {col.shape}")

print("\nPlot limits:")
print(f"Time range: 0 to {KEOGRAM_TIME_MIN_MAX} min")
print(f"Distance range: {KEOGRAM_DISTANCE_MIN} to {KEOGRAM_DISTANCE_MAX} km")

Selected rows:
-250 km row index: 250, distance = -250 km
   0 km row index: 500, distance = 0 km
 250 km row index: 750, distance = 250 km

Selected columns:
   0.0 min -> frame 0, actual time = 0.00 min
 100.0 min -> frame 50, actual time = 100.00 min
 150.0 min -> frame 75, actual time = 150.00 min
 200.0 min -> frame 100, actual time = 200.00 min

Saved:
BLO_perturbation_keogram_outputs_new\BLO_keogram_040_with_selected_rows_and_columns.png
BLO_perturbation_keogram_outputs_new\BLO_keogram_040_minus250km_row.png
BLO_perturbation_keogram_outputs_new\BLO_keogram_040_0km_row.png
BLO_perturbation_keogram_outputs_new\BLO_keogram_040_250km_row.png
BLO_perturbation_keogram_outputs_new\BLO_keogram_040_profiles_0_100_150_200min.png

Keogram shape:
time x distance = (280, 1001)

Extracted row arrays:
row_neg250 shape = (280,)
row_0km shape = (280,)
row_pos250 shape = (280,)

Extracted column arrays:
0 min column shape = (1001,)
100 min column shape = (1001,)
150 min column shape = (1001,)
200

## Overlay
Add another first row figure where each selected vertical profile is converted into an equivalent time span of 25 minutes, then overlaid at its correct center time: 0, 100, 150, and 200 min.

In [30]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path


# ==================================================
# Settings
# ==================================================
OUT_DIR = Path("BLO_perturbation_keogram_outputs_new")

ANGLE_DEG = 50

KEOGRAM_FILE = OUT_DIR / f"BLO_keogram_angle_{ANGLE_DEG:03d}.npy"
TIMES_FILE = OUT_DIR / "times_s.npy"

DPI = 200
CMAP = "RdBu_r"

TIMES_TO_PLOT = [0, 50, 100, 125, 150, 200, 250]

KEOGRAM_TIME_MIN_MAX = 300
KEOGRAM_DISTANCE_MIN = -250
KEOGRAM_DISTANCE_MAX = 250

PROFILE_TOTAL_TIME_MIN = 15.0


# ==================================================
# Read keogram
# ==================================================
keogram = np.load(KEOGRAM_FILE)
times_s = np.load(TIMES_FILE)
times_min = times_s / 60.0

nt, nspace = keogram.shape

distance_km = np.arange(-(nspace // 2), nspace // 2 + 1)

if len(distance_km) != nspace:
    distance_km = np.arange(nspace) - nspace // 2


# ==================================================
# Select rows at -250 km, 0 km, and 250 km
# ==================================================
idx_neg250 = np.argmin(np.abs(distance_km - (-250)))
idx_0km = np.argmin(np.abs(distance_km - 0))
idx_pos250 = np.argmin(np.abs(distance_km - 250))

row_neg250 = keogram[:, idx_neg250]
row_0km = keogram[:, idx_0km]
row_pos250 = keogram[:, idx_pos250]


# ==================================================
# Select time columns
# ==================================================
column_indices = []
columns = []

for t in TIMES_TO_PLOT:
    idx = np.argmin(np.abs(times_min - t))
    column_indices.append(idx)
    columns.append(keogram[idx, :])


print("Selected rows:")
print(f"-250 km row index: {idx_neg250}, distance = {distance_km[idx_neg250]} km")
print(f"   0 km row index: {idx_0km}, distance = {distance_km[idx_0km]} km")
print(f" 250 km row index: {idx_pos250}, distance = {distance_km[idx_pos250]} km")

print("\nSelected columns:")
for t, idx in zip(TIMES_TO_PLOT, column_indices):
    print(f"{t:6.1f} min -> frame {idx}, actual time = {times_min[idx]:.2f} min")


# ==================================================
# Color limits
# ==================================================
vmax = np.nanpercentile(np.abs(keogram), 99)

if not np.isfinite(vmax) or vmax == 0:
    vmax = 1.0

vmin = -vmax

time_colors = [
    "black",
    "blue",
    "green",
    "purple",
    "orange",
    "brown",
    "magenta",
]


# ==================================================
# Plot keogram with selected rows and time columns
# ==================================================
fig, ax = plt.subplots(figsize=(9, 5))

im = ax.imshow(
    keogram.T,
    origin="lower",
    aspect="auto",
    cmap=CMAP,
    vmin=vmin,
    vmax=vmax,
    extent=[
        times_min.min(),
        times_min.max(),
        distance_km.min(),
        distance_km.max(),
    ],
)

cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label("Perturbation amplitude")

ax.axhline(
    distance_km[idx_neg250],
    color="black",
    linewidth=2.0,
    label="-250 km",
)

ax.axhline(
    distance_km[idx_0km],
    color="gray",
    linewidth=2.0,
    linestyle="--",
    label="0 km",
)

ax.axhline(
    distance_km[idx_pos250],
    color="red",
    linewidth=2.0,
    label="250 km",
)

for t, idx, color in zip(TIMES_TO_PLOT, column_indices, time_colors):
    ax.axvline(
        times_min[idx],
        color=color,
        linewidth=1.8,
        linestyle=":",
        label=f"{t:.0f} min",
    )

ax.set_xlim(0, KEOGRAM_TIME_MIN_MAX)
ax.set_ylim(KEOGRAM_DISTANCE_MIN, KEOGRAM_DISTANCE_MAX)

ax.set_title(f"{ANGLE_DEG}° BLO Directional Keogram with Selected Rows and Time Columns")
ax.set_xlabel("Time (min)")
ax.set_ylabel("Distance along slice (km)")
ax.legend(loc="upper right", fontsize=8)

plt.tight_layout()

out_keogram_plot = OUT_DIR / f"BLO_keogram_{ANGLE_DEG:03d}_with_selected_rows_and_columns.png"
fig.savefig(out_keogram_plot, dpi=DPI, bbox_inches="tight")
plt.close(fig)


# ==================================================
# Plot -250 km row
# ==================================================
fig_neg250, ax_neg250 = plt.subplots(figsize=(8, 4))

ax_neg250.plot(times_min, row_neg250, linewidth=1.5)

ax_neg250.set_xlim(0, KEOGRAM_TIME_MIN_MAX)

ax_neg250.set_title(f"-250 km Row of {ANGLE_DEG}° BLO Keogram")
ax_neg250.set_xlabel("Time (min)")
ax_neg250.set_ylabel("Perturbation amplitude")
ax_neg250.grid(True, alpha=0.3)

plt.tight_layout()

out_neg250 = OUT_DIR / f"BLO_keogram_{ANGLE_DEG:03d}_minus250km_row.png"
fig_neg250.savefig(out_neg250, dpi=DPI, bbox_inches="tight")
plt.close(fig_neg250)


# ==================================================
# Overlay selected profiles onto -250 km row
# ==================================================
distance_mask = (
    (distance_km >= KEOGRAM_DISTANCE_MIN)
    & (distance_km <= KEOGRAM_DISTANCE_MAX)
)

distance_for_overlay = distance_km[distance_mask]

# Map distance to time offset:
# -250 km -> selected time
#    0 km -> selected time + 12.5 min
# +250 km -> selected time + 25.0 min
distance_norm = (
    distance_for_overlay - KEOGRAM_DISTANCE_MIN
) / (
    KEOGRAM_DISTANCE_MAX - KEOGRAM_DISTANCE_MIN
)

fig_neg250_overlay, ax_neg250_overlay = plt.subplots(figsize=(9, 4))

ax_neg250_overlay.plot(
    times_min,
    row_neg250,
    color="black",
    linewidth=1.5,
    label="-250 km row",
)

for t, idx, col, color in zip(TIMES_TO_PLOT, column_indices, columns, time_colors):
    profile_time = times_min[idx] + distance_norm * PROFILE_TOTAL_TIME_MIN
    profile_amp = col[distance_mask]

    ax_neg250_overlay.plot(
        profile_time,
        profile_amp,
        color=color,
        linewidth=1.5,
        alpha=0.8,
        label=f"profile starting at {t:.0f} min",
    )

    ax_neg250_overlay.axvline(
        times_min[idx],
        color=color,
        linestyle=":",
        linewidth=1.2,
        alpha=0.8,
    )

ax_neg250_overlay.set_xlim(0, KEOGRAM_TIME_MIN_MAX)

ax_neg250_overlay.set_title(
    f"-250 km Row of {ANGLE_DEG}° Keogram with Selected Profiles Overlaid\n"
    f"Profiles Start at Selected Times; -250 to 250 km Mapped to {PROFILE_TOTAL_TIME_MIN:.0f} min"
)
ax_neg250_overlay.set_xlabel("Time (min)")
ax_neg250_overlay.set_ylabel("Perturbation amplitude")
ax_neg250_overlay.grid(True, alpha=0.3)
ax_neg250_overlay.legend(loc="best", fontsize=8)

plt.tight_layout()

out_neg250_overlay = (
    OUT_DIR
    / f"BLO_keogram_{ANGLE_DEG:03d}_minus250km_row_with_profiles_starting_{PROFILE_TOTAL_TIME_MIN:.0f}min_mapping.png"
)
fig_neg250_overlay.savefig(out_neg250_overlay, dpi=DPI, bbox_inches="tight")
plt.close(fig_neg250_overlay)


# ==================================================
# Plot 0-km row
# ==================================================
fig_0, ax_0 = plt.subplots(figsize=(8, 4))

ax_0.plot(times_min, row_0km, linewidth=1.5)

ax_0.set_xlim(0, KEOGRAM_TIME_MIN_MAX)

ax_0.set_title(f"0-km Row of {ANGLE_DEG}° BLO Keogram")
ax_0.set_xlabel("Time (min)")
ax_0.set_ylabel("Perturbation amplitude")
ax_0.grid(True, alpha=0.3)

plt.tight_layout()

out_0 = OUT_DIR / f"BLO_keogram_{ANGLE_DEG:03d}_0km_row.png"
fig_0.savefig(out_0, dpi=DPI, bbox_inches="tight")
plt.close(fig_0)


# ==================================================
# Plot 250 km row
# ==================================================
fig_pos250, ax_pos250 = plt.subplots(figsize=(8, 4))

ax_pos250.plot(times_min, row_pos250, linewidth=1.5)

ax_pos250.set_xlim(0, KEOGRAM_TIME_MIN_MAX)

ax_pos250.set_title(f"250 km Row of {ANGLE_DEG}° BLO Keogram")
ax_pos250.set_xlabel("Time (min)")
ax_pos250.set_ylabel("Perturbation amplitude")
ax_pos250.grid(True, alpha=0.3)

plt.tight_layout()

out_pos250 = OUT_DIR / f"BLO_keogram_{ANGLE_DEG:03d}_250km_row.png"
fig_pos250.savefig(out_pos250, dpi=DPI, bbox_inches="tight")
plt.close(fig_pos250)


# ==================================================
# Plot selected time columns/profiles
# ==================================================
fig_c, ax_c = plt.subplots(figsize=(6, 7))

for t, idx, col, color in zip(TIMES_TO_PLOT, column_indices, columns, time_colors):
    ax_c.plot(
        col,
        distance_km,
        linewidth=1.5,
        color=color,
        label=f"{t:.0f} min",
    )

ax_c.axhline(
    distance_km[idx_neg250],
    color="black",
    linestyle="--",
    linewidth=1.2,
    label="-250 km",
)

ax_c.axhline(
    distance_km[idx_0km],
    color="gray",
    linestyle="--",
    linewidth=1.2,
    label="0 km",
)

ax_c.axhline(
    distance_km[idx_pos250],
    color="red",
    linestyle="--",
    linewidth=1.2,
    label="250 km",
)

ax_c.set_ylim(KEOGRAM_DISTANCE_MIN, KEOGRAM_DISTANCE_MAX)

ax_c.set_title(f"Profiles Along {ANGLE_DEG}° Slice at Selected Times")
ax_c.set_xlabel("Perturbation amplitude")
ax_c.set_ylabel("Distance along slice (km)")
ax_c.grid(True, alpha=0.3)
ax_c.legend(loc="best", fontsize=8)

plt.tight_layout()

time_label = "_".join([f"{int(t)}" for t in TIMES_TO_PLOT])
out_columns = OUT_DIR / f"BLO_keogram_{ANGLE_DEG:03d}_profiles_{time_label}min.png"
fig_c.savefig(out_columns, dpi=DPI, bbox_inches="tight")
plt.close(fig_c)


# ==================================================
# Print outputs
# ==================================================
print("\nSaved:")
print(out_keogram_plot)
print(out_neg250)
print(out_neg250_overlay)
print(out_0)
print(out_pos250)
print(out_columns)

print("\nKeogram shape:")
print(f"time x distance = {keogram.shape}")

print("\nExtracted row arrays:")
print(f"row_neg250 shape = {row_neg250.shape}")
print(f"row_0km shape = {row_0km.shape}")
print(f"row_pos250 shape = {row_pos250.shape}")

print("\nExtracted column arrays:")
for t, col in zip(TIMES_TO_PLOT, columns):
    print(f"{t:.0f} min column shape = {col.shape}")

print("\nPlot limits:")
print(f"Time range: 0 to {KEOGRAM_TIME_MIN_MAX} min")
print(f"Distance range: {KEOGRAM_DISTANCE_MIN} to {KEOGRAM_DISTANCE_MAX} km")

print("\nOverlay mapping:")
print(f"{KEOGRAM_DISTANCE_MIN} km -> selected time")
print(f"0 km -> selected time + {PROFILE_TOTAL_TIME_MIN / 2:.1f} min")
print(f"{KEOGRAM_DISTANCE_MAX} km -> selected time + {PROFILE_TOTAL_TIME_MIN:.1f} min")

Selected rows:
-250 km row index: 250, distance = -250 km
   0 km row index: 500, distance = 0 km
 250 km row index: 750, distance = 250 km

Selected columns:
   0.0 min -> frame 0, actual time = 0.00 min
  50.0 min -> frame 25, actual time = 50.00 min
 100.0 min -> frame 50, actual time = 100.00 min
 125.0 min -> frame 62, actual time = 124.00 min
 150.0 min -> frame 75, actual time = 150.00 min
 200.0 min -> frame 100, actual time = 200.00 min
 250.0 min -> frame 125, actual time = 250.00 min

Saved:
BLO_perturbation_keogram_outputs_new\BLO_keogram_050_with_selected_rows_and_columns.png
BLO_perturbation_keogram_outputs_new\BLO_keogram_050_minus250km_row.png
BLO_perturbation_keogram_outputs_new\BLO_keogram_050_minus250km_row_with_profiles_starting_15min_mapping.png
BLO_perturbation_keogram_outputs_new\BLO_keogram_050_0km_row.png
BLO_perturbation_keogram_outputs_new\BLO_keogram_050_250km_row.png
BLO_perturbation_keogram_outputs_new\BLO_keogram_050_profiles_0_50_100_125_150_200_250min.p

### Calculate speed: 
650 km in 1500 s => 433 m/s

650 km is 25/120 of wavelength. => wavelength = 650 km /(25/120) = 3120 km. 